# Simple Parameter Sensitivity

This notebook changes one behavioral or demand parameter group at a time and plots one chosen relative metric.

Metrics available:

- `average_utilization`: average daily utilization from served patients only, excluding no-shows
- `mean_accepted_booking_delay`: average booking delay among patients who accepted/booked a slot
- `mean_offered_booking_delay`: average offered delay among patients who received an offer, including those who balked
- `overall_percent_serviced`: aggregate served patients / aggregate arrivals

There are no total-served or total-value comparisons here because those scale with `measure_days`. Rates and averages are comparable when the simulation length changes.

For balking and no-show threshold rules, the notebook compares class 1 against class 2 directly:

- class 1 step size vs class 2 step size
- class 1 threshold vs class 2 threshold

For arrival-mix scenarios, total arrival rate is `lambda_total`, class 1 gets `p * lambda_total`, and class 2 gets `(1 - p) * lambda_total`.

The class arrival-rate section varies one class's arrival rate at a time, keeps the other class fixed, and gives two plots per class:

- rate plot: aggregate average utilization and that class's percent serviced
- delay plot: that class's mean accepted and mean offered booking delay

In [ ]:
from pathlib import Path
from dataclasses import replace
from itertools import product
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_repo_dir(start):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs" / "baseline.yaml").exists() and (
            candidate / "simulation" / "config_loader.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import class_result_rows, result_metrics_from_result
from analysis.plot_style import ARRIVAL_COLOR, driver_from_text, driver_heatmap_cmap, plot_driver_line
from simulation.config_loader import load_config
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import ThresholdRule

plt.style.use("default")

In [ ]:
base_config = load_config(REPO_DIR / "configs" / "baseline.yaml")

# Change this between the metric names listed above to plot different heatmaps.
# METRIC = "mean_offered_booking_delay"
METRIC = "average_utilization"

STEPS = np.linspace(0, 1, 21)
THRESHOLDS = range(base_config.horizon_days)
CANCEL_PROBS = np.linspace(0, 0.30, 14)

BASE_LAMBDA_TOTAL = sum(params.lambda_per_day for params in base_config.classes.values())
ARRIVAL_RATE_MULTIPLIERS = np.linspace(0.5, 1.5, 12)
CLASS_1_SHARES = np.linspace(0.1, 0.9, 18)

SEED = 123

## Metric Code

In [ ]:
def run_metrics(config, seed=SEED):
    config = replace(config, seed=seed)
    results = ClinicAppointmentSimulation(config).run()
    return {
        **result_metrics_from_result(results),
        "booked_slots": results.slot_metrics.booked_slots,
        "served_slots": results.slot_metrics.served_slots,
        "no_show_slots": results.slot_metrics.no_show_slots,
    }

base_metrics = pd.DataFrame([run_metrics(base_config)])
display(base_metrics)

## Small Helpers To Change Parameters

In [ ]:
def make_step_rule(old_rule, threshold=None, step=None):
    threshold = old_rule.threshold if threshold is None else int(threshold)
    low = old_rule.low
    step = old_rule.high - old_rule.low if step is None else float(step)
    return ThresholdRule(threshold=threshold, low=low, high=min(low + step, 1.0))


def update_classes(config, changes):
    classes = {}
    for class_id, params in config.classes.items():
        classes[class_id] = replace(params, **changes.get(class_id, {}))
    return replace(config, classes=classes)


def set_balking(config, class_steps=None, class_thresholds=None):
    class_steps = class_steps or {}
    class_thresholds = class_thresholds or {}
    changes = {}
    for class_id, params in config.classes.items():
        changes[class_id] = {
            "balk_prob": make_step_rule(
                params.balk_prob,
                threshold=class_thresholds.get(class_id),
                step=class_steps.get(class_id),
            )
        }
    return update_classes(config, changes)


def set_no_show(config, class_steps=None, class_thresholds=None):
    class_steps = class_steps or {}
    class_thresholds = class_thresholds or {}
    changes = {}
    for class_id, params in config.classes.items():
        changes[class_id] = {
            "no_show_prob": make_step_rule(
                params.no_show_prob,
                threshold=class_thresholds.get(class_id),
                step=class_steps.get(class_id),
            )
        }
    return update_classes(config, changes)


def set_cancellation(config, class_probs):
    changes = {class_id: {"cancel_prob": prob} for class_id, prob in class_probs.items()}
    return update_classes(config, changes)


def set_arrival_mix(config, lambda_total, class_1_share):
    class_1_share = float(class_1_share)
    lambda_total = float(lambda_total)
    return update_classes(
        config,
        {
            1: {"lambda_per_day": class_1_share * lambda_total},
            2: {"lambda_per_day": (1 - class_1_share) * lambda_total},
        },
    )


def set_arrival_multiplier(config, multiplier):
    lambda_total = BASE_LAMBDA_TOTAL * float(multiplier)
    base_class_1_share = config.classes[1].lambda_per_day / BASE_LAMBDA_TOTAL
    return set_arrival_mix(config, lambda_total, base_class_1_share)


def set_class_arrival_rate(config, class_id, lambda_per_day):
    return update_classes(config, {class_id: {"lambda_per_day": float(lambda_per_day)}})


def run_class_metrics(config, seed=SEED):
    config = replace(config, seed=seed)
    results = ClinicAppointmentSimulation(config).run()
    df = pd.DataFrame(
        class_result_rows(
            results,
            {
                "average_utilization": results.average_utilization,
                "overall_percent_serviced": results.overall_percent_serviced,
            },
        )
    )
    df["lambda_per_day"] = df["class_id"].map(lambda class_id: config.classes[class_id].lambda_per_day)
    return df

In [ ]:
def heatmap(df, x, y, value=METRIC, title=None, driver=None):
    table = df.pivot(index=y, columns=x, values=value).sort_index().sort_index(axis=1)
    driver = driver or driver_from_text(x, y, title, value)

    fig, ax = plt.subplots(figsize=(7, 5))
    image = ax.imshow(table.values, origin="lower", aspect="auto", cmap=driver_heatmap_cmap(driver))
    ax.set_title(title or value)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels([f"{v:.2f}" if isinstance(v, float) else v for v in table.columns])
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels([f"{v:.2f}" if isinstance(v, float) else v for v in table.index])
    fig.colorbar(image, ax=ax, label=value)
    plt.show()

    return table

## 1. Balking Step Size By Class

Everything stays at baseline except the balking step size for class 1 and class 2.

In [ ]:
rows = []
for class_1_step, class_2_step in product(STEPS, STEPS):
    config = set_balking(base_config, class_steps={1: class_1_step, 2: class_2_step})
    rows.append({
        "class_1_step": class_1_step,
        "class_2_step": class_2_step,
        **run_metrics(config),
    })

balk_class_df = pd.DataFrame(rows)
heatmap(balk_class_df, "class_1_step", "class_2_step", title=f"Balking step by class: {METRIC}")

## 2. Balking Threshold By Class

Everything stays at baseline except the balking threshold for class 1 and class 2. Step sizes stay fixed at baseline.

In [ ]:
rows = []
for class_1_threshold, class_2_threshold in product(THRESHOLDS, THRESHOLDS):
    config = set_balking(
        base_config,
        class_thresholds={1: class_1_threshold, 2: class_2_threshold},
    )
    rows.append({
        "class_1_threshold": class_1_threshold,
        "class_2_threshold": class_2_threshold,
        **run_metrics(config),
    })

balk_threshold_df = pd.DataFrame(rows)
heatmap(
    balk_threshold_df,
    "class_1_threshold",
    "class_2_threshold",
    title=f"Balking threshold by class: {METRIC}",
)

## 3. Balking Threshold And Jump Level

Both classes use the same balking threshold and jump level. This isolates the absolute effect of the jump moment and jump size, instead of class-to-class asymmetry.

In [ ]:
rows = []
for threshold, jump_level in product(THRESHOLDS, STEPS):
    config = set_balking(
        base_config,
        class_steps={1: jump_level, 2: jump_level},
        class_thresholds={1: threshold, 2: threshold},
    )
    rows.append({
        "threshold": threshold,
        "jump_level": jump_level,
        **run_metrics(config),
    })

balk_threshold_jump_df = pd.DataFrame(rows)
heatmap(
    balk_threshold_jump_df,
    "jump_level",
    "threshold",
    title=f"Balking threshold and jump level: {METRIC}",
)

## 4. No-Show Step Size By Class

Everything stays at baseline except the no-show step size for class 1 and class 2. Thresholds stay fixed at baseline.

In [ ]:
rows = []
for class_1_step, class_2_step in product(STEPS, STEPS):
    config = set_no_show(base_config, class_steps={1: class_1_step, 2: class_2_step})
    rows.append({
        "class_1_step": class_1_step,
        "class_2_step": class_2_step,
        **run_metrics(config),
    })

no_show_step_df = pd.DataFrame(rows)
heatmap(
    no_show_step_df,
    "class_1_step",
    "class_2_step",
    title=f"No-show step by class: {METRIC}",
)

## 5. No-Show Threshold By Class

Everything stays at baseline except the no-show threshold for class 1 and class 2. Step sizes stay fixed at baseline.

In [ ]:
rows = []
for class_1_threshold, class_2_threshold in product(THRESHOLDS, THRESHOLDS):
    config = set_no_show(
        base_config,
        class_thresholds={1: class_1_threshold, 2: class_2_threshold},
    )
    rows.append({
        "class_1_threshold": class_1_threshold,
        "class_2_threshold": class_2_threshold,
        **run_metrics(config),
    })

no_show_threshold_df = pd.DataFrame(rows)
heatmap(
    no_show_threshold_df,
    "class_1_threshold",
    "class_2_threshold",
    title=f"No-show threshold by class: {METRIC}",
)

## 6. No-Show Threshold And Jump Level

Both classes use the same no-show threshold and jump level. This isolates the absolute effect of the no-show jump moment and jump size.

In [ ]:
rows = []
for threshold, jump_level in product(THRESHOLDS, STEPS):
    config = set_no_show(
        base_config,
        class_steps={1: jump_level, 2: jump_level},
        class_thresholds={1: threshold, 2: threshold},
    )
    rows.append({
        "threshold": threshold,
        "jump_level": jump_level,
        **run_metrics(config),
    })

no_show_threshold_jump_df = pd.DataFrame(rows)
heatmap(
    no_show_threshold_jump_df,
    "jump_level",
    "threshold",
    title=f"No-show threshold and jump level: {METRIC}",
)

## 7. Cancellation Probability By Class

Cancellation has no threshold in this model, so this varies the daily cancellation probability for each class.

In [ ]:
rows = []
for class_1_cancel, class_2_cancel in product(CANCEL_PROBS, CANCEL_PROBS):
    config = set_cancellation(base_config, {1: class_1_cancel, 2: class_2_cancel})
    rows.append({
        "class_1_cancel": class_1_cancel,
        "class_2_cancel": class_2_cancel,
        **run_metrics(config),
    })

cancel_df = pd.DataFrame(rows)
heatmap(cancel_df, "class_1_cancel", "class_2_cancel", title=f"Cancellation by class: {METRIC}")

## 8. Total Arrival Rate And Class Mix

This varies demand pressure and the class mix together. For each grid point:

- `lambda_total = multiplier * baseline lambda_total`
- `lambda_1 = p * lambda_total`
- `lambda_2 = (1 - p) * lambda_total`

All behavioral rules stay fixed at baseline.

In [ ]:
rows = []
for multiplier, class_1_share in product(ARRIVAL_RATE_MULTIPLIERS, CLASS_1_SHARES):
    lambda_total = BASE_LAMBDA_TOTAL * multiplier
    config = set_arrival_mix(base_config, lambda_total, class_1_share)
    rows.append({
        "arrival_multiplier": multiplier,
        "class_1_share": class_1_share,
        "lambda_total": lambda_total,
        "lambda_1": class_1_share * lambda_total,
        "lambda_2": (1 - class_1_share) * lambda_total,
        **run_metrics(config),
    })

demand_mix_df = pd.DataFrame(rows)
heatmap(
    demand_mix_df,
    "class_1_share",
    "arrival_multiplier",
    title=f"Arrival rate and class mix: {METRIC}",
)

## 9. Class Arrival Rate Sweeps

These plots vary one class's arrival rate at a time and keep the other class fixed at baseline.

For each class there are two plots against that class's arrival rate:

- aggregate average utilization and that class's percent serviced
- that class's mean accepted and mean offered booking delay

In [ ]:
rows = []
for target_class in sorted(base_config.classes):
    base_lambda = base_config.classes[target_class].lambda_per_day
    for multiplier in ARRIVAL_RATE_MULTIPLIERS:
        class_lambda = base_lambda * multiplier
        config = set_class_arrival_rate(base_config, target_class, class_lambda)
        class_metrics = run_class_metrics(config)
        row = class_metrics.loc[class_metrics["class_id"].eq(target_class)].iloc[0].to_dict()
        rows.append({
            "target_class": target_class,
            "arrival_multiplier": multiplier,
            "target_lambda": class_lambda,
            **row,
        })

class_arrival_df = pd.DataFrame(rows)
display(class_arrival_df.head())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

for row_idx, target_class in enumerate(sorted(base_config.classes)):
    data = class_arrival_df.loc[class_arrival_df["target_class"].eq(target_class)].sort_values("target_lambda")

    rate_ax = axes[row_idx, 0]
    plot_driver_line(rate_ax, data["target_lambda"], data["average_utilization"], "average utilization", driver="arrival", index=0)
    plot_driver_line(rate_ax, data["target_lambda"], data["percent_serviced"], "class percent serviced", driver="arrival", index=target_class)
    rate_ax.set_title(f"Class {target_class}: rates vs arrival rate")
    rate_ax.set_xlabel("arrival rate lambda")
    rate_ax.set_ylabel("rate")
    rate_ax.legend(frameon=False)

    delay_ax = axes[row_idx, 1]
    plot_driver_line(delay_ax, data["target_lambda"], data["mean_accepted_booking_delay"], "accepted delay", driver="arrival", index=0)
    plot_driver_line(delay_ax, data["target_lambda"], data["mean_offered_booking_delay"], "offered delay", driver="arrival", index=target_class)
    delay_ax.set_title(f"Class {target_class}: booking delay vs arrival rate")
    delay_ax.set_xlabel("arrival rate lambda")
    delay_ax.set_ylabel("booking delay")
    delay_ax.legend(frameon=False)

plt.show()

## 10. Named Demand Scenarios

These are a few readable scenarios around the baseline. They are useful when you want a compact table rather than a full grid.

In [ ]:
named_scenarios = [
    ("baseline", 1.00, base_config.classes[1].lambda_per_day / BASE_LAMBDA_TOTAL),
    ("low demand", 0.75, base_config.classes[1].lambda_per_day / BASE_LAMBDA_TOTAL),
    ("high demand", 1.25, base_config.classes[1].lambda_per_day / BASE_LAMBDA_TOTAL),
    ("class 1 heavy", 1.00, 0.75),
    ("class 2 heavy", 1.00, 0.25),
    ("high demand, class 1 heavy", 1.25, 0.75),
    ("high demand, class 2 heavy", 1.25, 0.25),
]

rows = []
for name, multiplier, class_1_share in named_scenarios:
    lambda_total = BASE_LAMBDA_TOTAL * multiplier
    config = set_arrival_mix(base_config, lambda_total, class_1_share)
    rows.append({
        "scenario": name,
        "arrival_multiplier": multiplier,
        "class_1_share": class_1_share,
        "lambda_1": class_1_share * lambda_total,
        "lambda_2": (1 - class_1_share) * lambda_total,
        **run_metrics(config),
    })

named_demand_df = pd.DataFrame(rows)
display(named_demand_df.sort_values(METRIC, ascending=(METRIC == "mean_accepted_booking_delay")))

fig, ax = plt.subplots(figsize=(9, 4))
named_demand_df.plot(kind="bar", x="scenario", y=METRIC, ax=ax, legend=False, color=ARRIVAL_COLOR)
ax.set_title(f"Named demand scenarios: {METRIC}")
ax.set_xlabel("")
ax.set_ylabel(METRIC)
plt.xticks(rotation=35, ha="right")
plt.show()

## Best Tested Settings

In [ ]:
def best_rows(df, n=5):
    lower_is_better = {
        "mean_accepted_booking_delay",
        "mean_offered_booking_delay",
    }
    return df.sort_values(METRIC, ascending=METRIC in lower_is_better).head(n)

print("Balking step sweep")
display(best_rows(balk_class_df))

print("Balking threshold sweep")
display(best_rows(balk_threshold_df))

print("Balking threshold and jump-level sweep")
display(best_rows(balk_threshold_jump_df))

print("No-show step sweep")
display(best_rows(no_show_step_df))

print("No-show threshold sweep")
display(best_rows(no_show_threshold_df))

print("No-show threshold and jump-level sweep")
display(best_rows(no_show_threshold_jump_df))

print("Cancellation sweep")
display(best_rows(cancel_df))

print("Arrival rate and class-mix sweep")
display(best_rows(demand_mix_df))

print("Named demand scenarios")
display(best_rows(named_demand_df))

print("Class arrival-rate sweeps")
display(class_arrival_df.sort_values(["target_class", "target_lambda"]))